# SAFE rollouts：Google Drive → Colab → 服务器

数据在云端直接流式写入服务器，不保存到本地电脑，也不占用本地下载流量。密码仅通过隐藏输入进入当前 Colab 内存，不写入代码或文件。

In [ ]:
!pip -q install 'gdown>=5.2.0' 'paramiko>=3.5.0'

In [ ]:
import base64
import getpass
import hashlib
import socket
import stat

import gdown
import paramiko

HOST = 'connect.bjb2.seetacloud.com'
PORT = 20559
USER = 'root'
REMOTE_DIR = '/root/autodl-tmp/datasets/safe-rollouts'
EXPECTED_HOST_KEY = 'SHA256:liZ36vNCsNcNdXeWs4f+g5ZIhPM/ZihP834vxs8Ulqc'
FILES = [
    ('13z_cdwnaJota2iHkZbhYgVALujZwtM3b', 'pi0fast_droid_0510_all.zip'),
    ('1EwaccasZjnlM9L6SEYyWqTd7d6-BR9zp', 'openvla_widowx.zip'),
]

password = getpass.getpass('服务器密码（输入不会显示）：')
sock = socket.create_connection((HOST, PORT), timeout=30)
transport = paramiko.Transport(sock)
transport.start_client(timeout=30)
server_key = transport.get_remote_server_key()
fingerprint = 'SHA256:' + base64.b64encode(
    hashlib.sha256(server_key.asbytes()).digest()
).decode().rstrip('=')
if fingerprint != EXPECTED_HOST_KEY:
    transport.close()
    raise RuntimeError(f'服务器指纹不匹配：{fingerprint}')
transport.auth_password(USER, password)
password = None
sftp = paramiko.SFTPClient.from_transport(transport)

def mkdir_p(path):
    current = ''
    for part in path.strip('/').split('/'):
        current += '/' + part
        try:
            mode = sftp.stat(current).st_mode
            if not stat.S_ISDIR(mode):
                raise RuntimeError(f'远程路径不是目录：{current}')
        except FileNotFoundError:
            sftp.mkdir(current)

mkdir_p(REMOTE_DIR)

try:
    for file_id, filename in FILES:
        final_path = f'{REMOTE_DIR}/{filename}'
        part_path = final_path + '.part'
        try:
            size = sftp.stat(final_path).st_size
            print(f'已存在，跳过：{final_path} ({size:,} bytes)')
            continue
        except FileNotFoundError:
            pass

        print(f'开始云端传输：{filename}')
        remote_file = sftp.file(part_path, mode='wb', bufsize=1024 * 1024)
        remote_file.set_pipelined(True)
        try:
            gdown.download(
                id=file_id,
                output=remote_file,
                quiet=False,
                use_cookies=True,
            )
            remote_file.flush()
        finally:
            remote_file.close()

        size = sftp.stat(part_path).st_size
        if size == 0:
            raise RuntimeError(f'下载结果为空：{filename}')
        sftp.rename(part_path, final_path)
        print(f'完成：{final_path} ({size:,} bytes)')
finally:
    sftp.close()
    transport.close()

print('两份数据传输结束。请断开并删除当前 Colab 运行时。')